In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

pd.set_option('display.max_columns', 200)

# Load the canonical clean dataset
df = pd.read_parquet('../data/loans_clean.parquet')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
print(f"Default rate: {df['default'].mean():.4f}")

In [ ]:
# Apply data cleaning from the EDA findings (Section 2 of reports/eda_findings.md)
df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

print("Data cleaning applied")
print(f"\nQuick check on cleaned features:")
print(df[['annual_inc', 'dti', 'revol_util', 'credit_history_months']].describe(
    percentiles=[0.99]
).round(2))

In [ ]:
# Temporal split (decision from Day 2 — 2018 maturity bias finding)
train_mask = df['issue_year'].isin([2014, 2015])
val_mask = df['issue_year'] == 2016
test_mask = df['issue_year'] == 2017

df_train = df[train_mask].copy()
df_val = df[val_mask].copy()
df_test = df[test_mask].copy()

print(f"Train (2014-2015): {len(df_train):>9,} loans   default rate {df_train['default'].mean():.4f}")
print(f"Val   (2016):      {len(df_val):>9,} loans   default rate {df_val['default'].mean():.4f}")
print(f"Test  (2017):      {len(df_test):>9,} loans   default rate {df_test['default'].mean():.4f}")
print(f"\nTotal used:        {len(df_train) + len(df_val) + len(df_test):>9,} loans")
print(f"Total in dataset:  {len(df):>9,} loans")
print(f"Dropped (2018 + pre-2014): {len(df) - len(df_train) - len(df_val) - len(df_test):,} loans")

In [ ]:
# Cost matrix (rough but realistic assumptions):
# - Good loan (Fully Paid):  earn ~10% of loan_amnt in net interest over loan life
# - Bad loan (Charged Off):  lose ~50% of loan_amnt (assumes ~50% recovery)
# - Rejected loan:           $0 (simplification, ignores opportunity cost)

GAIN_RATE = 0.10
LOSS_RATE = 0.50

def calculate_profit(df_subset, approval_mask):
    """Total profit from a set of approval decisions."""
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved['default'] == 0,
         GAIN_RATE * approved['loan_amnt'],   # good loan: earn
        -LOSS_RATE * approved['loan_amnt']    # bad loan: lose
    )
    return profit.sum()

print(f"Cost matrix:")
print(f"  Good loan: +{GAIN_RATE:.0%} of loan_amnt")
print(f"  Bad loan:  -{LOSS_RATE:.0%} of loan_amnt")
print(f"  Rejected:  $0")

In [ ]:
# Baseline 1: approve every applicant, no prediction
# This is the do-nothing floor

y_val = df_val['default'].values

# For AUC purposes, use constant probability = train default rate
probs_approve_all = np.full(len(df_val), df_train['default'].mean())

# Profit: approve every val loan
profit_approve_all = calculate_profit(df_val, np.ones(len(df_val), dtype=bool))

print("=" * 55)
print("BASELINE 1: Approve everyone")
print("=" * 55)
print(f"AUC:        0.5000 (by definition — no discrimination)")
print(f"Brier:      {brier_score_loss(y_val, probs_approve_all):.4f}")
print(f"\nApproved:   {len(df_val):,} loans (100%)")
print(f"Defaulters: {int(y_val.sum()):,} ({y_val.mean():.2%})")
print(f"\nTotal profit:   ${profit_approve_all:>14,.0f}")
print(f"Per loan:       ${profit_approve_all / len(df_val):>14,.2f}")

In [ ]:
# Baseline 2: use LC's grade (A-G) as the predictor
# Learn default rate per grade from TRAIN ONLY (no peeking at val)

grade_default_rates = df_train.groupby('grade')['default'].mean().sort_index()
print("Grade → Predicted P(default), learned from TRAIN ONLY:")
print(grade_default_rates.round(4))

# Apply to val
val_grade_probs = df_val['grade'].map(grade_default_rates).values

# Metrics
auc = roc_auc_score(y_val, val_grade_probs)
pr_auc = average_precision_score(y_val, val_grade_probs)
brier = brier_score_loss(y_val, val_grade_probs)

print(f"\n{'=' * 55}")
print(f"BASELINE 2: Grade only")
print(f"{'=' * 55}")
print(f"AUC:      {auc:.4f}")
print(f"PR-AUC:   {pr_auc:.4f}")
print(f"Brier:    {brier:.4f}")

In [ ]:
# Find the threshold that maximizes profit
thresholds = np.linspace(0.05, 0.55, 100)
profits, approval_rates, defaulter_rejection = [], [], []

for threshold in thresholds:
    # Approve if predicted default prob is BELOW threshold
    approval_mask = val_grade_probs < threshold
    profits.append(calculate_profit(df_val, approval_mask))
    approval_rates.append(approval_mask.mean())
    # What fraction of defaulters did we correctly reject?
    rejected_defaulters = ((~approval_mask) & (y_val == 1)).sum()
    defaulter_rejection.append(rejected_defaulters / y_val.sum())

profits = np.array(profits)
best_idx = profits.argmax()

print(f"Grade-only baseline at optimal threshold:")
print(f"  Best threshold:           {thresholds[best_idx]:.4f}")
print(f"  Approval rate:            {approval_rates[best_idx]:.2%}")
print(f"  Defaulters rejected:      {defaulter_rejection[best_idx]:.2%}")
print(f"  Total profit at optimum:  ${profits[best_idx]:>14,.0f}")

print(f"\nComparison:")
print(f"  Approve-all profit:     ${profit_approve_all:>14,.0f}")
print(f"  Grade-only profit:      ${profits[best_idx]:>14,.0f}")
print(f"  Improvement:            ${profits[best_idx] - profit_approve_all:>14,.0f}")